# Notebook to design a method to find the Circoscription with an Address

## Step 1 : Normalize and Geolocate the address

API of the data base Adresse Nationale (BAN)

API Géoplateforme - Géocodage :
- user guide https://www.data.gouv.fr/dataservices/api-geoplateforme-geocodage
- swagger : https://data.geopf.fr/geocodage/openapi

In [3]:
#name of the server containing the geocoding service
server_name = "https://data.geopf.fr/geocodage"

## Search service of the Géoplateforme - Géocodage API
with GET Method

A noter : limite de 50 requêtes par seconde depuis une même adresse IP

In [4]:
#get method with requests library
import requests

#address to geocode
address = "2 rue henri bellevue, 65000 Tarbes"

response = requests.get(
    server_name + "/search", #endpoint of the geocoding service
    params={
        "q": address,
        "format": "json",
        "index": "address,poi", #index to search in, can be address or poi (point of interest or admin unit)
        "limit": "10" #max number of results to return
    }
)

test_address_geocode = response.json()

print(test_address_geocode)

{'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [0.062997, 43.237242]}, 'properties': {'label': '2 Rue Henri Bellevue 65000 Tarbes', 'score': 0.9675481818181817, 'housenumber': '2', 'id': '65440_1640_00002', 'banId': 'e0c04698-2e2f-4885-a266-4aa937afa381', 'name': '2 Rue Henri Bellevue', 'postcode': '65000', 'citycode': '65440', 'x': 461336.1, 'y': 6241999.55, 'city': 'Tarbes', 'context': '65, Hautes-Pyrénées, Occitanie', 'type': 'housenumber', 'importance': 0.64303, 'depcode': '65', 'street': 'Rue Henri Bellevue', '_type': 'address'}}], 'query': '2 rue henri bellevue, 65000 Tarbes'}


Print pretty the Json returned

In [5]:
import json

print(json.dumps(test_address_geocode, indent=4, ensure_ascii=False))

{
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [
                    0.062997,
                    43.237242
                ]
            },
            "properties": {
                "label": "2 Rue Henri Bellevue 65000 Tarbes",
                "score": 0.9675481818181817,
                "housenumber": "2",
                "id": "65440_1640_00002",
                "banId": "e0c04698-2e2f-4885-a266-4aa937afa381",
                "name": "2 Rue Henri Bellevue",
                "postcode": "65000",
                "citycode": "65440",
                "x": 461336.1,
                "y": 6241999.55,
                "city": "Tarbes",
                "context": "65, Hautes-Pyrénées, Occitanie",
                "type": "housenumber",
                "importance": 0.64303,
                "depcode": "65",
                "street": "Rue Henri Bellevue",
   

Retrieve coordinates

In [6]:
#from the response converted in json, we extract the coordinates
test_address_geocode_coordinates = test_address_geocode["features"][0]["geometry"]["coordinates"]

print(test_address_geocode_coordinates)

[0.062997, 43.237242]


## Step 2: Get the circo from the coordinates

### Method 1 : download locally GEOJson map and find polygon containing the coordinates

Reading the shp file donwloaded from https://www.data.gouv.fr/datasets/portraits-des-circonscriptions-legislatives-indicateurs-economiques-et-socio-demographiques

In [7]:
import geopandas as gpd

gpf = gpd.read_file("countours_files/circonscriptions_legislatives_030522.shp")

print(gpf.head())

  id_circo  dep                           libelle  \
0    97302  973       Guyane - 2e circonscription   
1    97301  973      Guyane - 1re circonscription   
2    97201  972  Martinique - 1re circonscription   
3    97202  972   Martinique - 2e circonscription   
4    97203  972   Martinique - 3e circonscription   

                                            geometry  
0  MULTIPOLYGON (((-54.60236 2.33356, -54.60242 2...  
1  MULTIPOLYGON (((-51.93691 4.46648, -51.93667 4...  
2  MULTIPOLYGON (((-61.00495 14.57791, -61.00492 ...  
3  MULTIPOLYGON (((-61.12889 14.63162, -61.1289 1...  
4  POLYGON ((-61.03945 14.64265, -61.0394 14.6425...  


In [8]:
from shapely.geometry import Point
from shapely.geometry.polygon import Polygon

point = Point(test_address_geocode_coordinates[0], test_address_geocode_coordinates[1])

for i in range(len(gpf)):
    polygon = gpf.loc[i, 'geometry']
    if polygon.contains(point):
        print(f"The point is inside the polygon with index {i}. The circonscription is {gpf.loc[i, 'libelle']}.")
        break

The point is inside the polygon with index 204. The circonscription is Hautes-Pyrénées - 1re circonscription.


### Method 2 : Use Service API to return circo with GPS coordinates

Services found so far : 
- Managed by SciencePO : https://services-eu1.arcgis.com/bFTOZwh6Wo1fgzl7/ArcGIS/rest/services/Circonscriptions_legislatives_France/FeatureServer
- Managed by SciencePO : https://services-eu1.arcgis.com/bFTOZwh6Wo1fgzl7/ArcGIS/rest/services/Circonscriptions_legislatives_FranceT1_T2/FeatureServer

For those 2 services, the method to identify the Circo seems to be the same as in method 1

## Step 3 : How to deal with multiple results sent by the Geocodage API

In [9]:
#address to geocode
address_multi = "30 rue Léon Frot"

response_multi = requests.get(
    server_name + "/search", #endpoint of the geocoding service
    params={
        "q": address_multi,
        "format": "json",
        "index": "address,poi", #index to search in, can be address or poi (point of interest or admin unit)
        "limit": "10" #max number of results to return
    }
)

test_address_multi_geocode = response_multi.json()

#print pretty json
print(json.dumps(test_address_multi_geocode, indent=4, ensure_ascii=False))


{
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [
                    2.387385,
                    48.854959
                ]
            },
            "properties": {
                "label": "30 Rue Léon Frot 75011 Paris",
                "score": 0.978830909090909,
                "housenumber": "30",
                "id": "75111_5554_00030",
                "banId": "397265f7-7916-422e-9066-702107f8437c",
                "name": "30 Rue Léon Frot",
                "postcode": "75011",
                "citycode": "75111",
                "x": 655049.18,
                "y": 6861832.19,
                "city": "Paris",
                "district": "Paris 11e Arrondissement",
                "context": "75, Paris, Île-de-France",
                "type": "housenumber",
                "importance": 0.76714,
                "depcode": "75",
          

In [10]:
# create dictionnary with labels and coordinates of the results
results_dict = {}

for i in range(len(test_address_multi_geocode["features"])):

    label = test_address_multi_geocode["features"][i]["properties"]["label"]
    coords = test_address_multi_geocode["features"][i]["geometry"]["coordinates"]
    
    results_dict[label] = coords

print("Results:", results_dict)


Results: {'30 Rue Léon Frot 75011 Paris': [2.387385, 48.854959], 'Rue Léon Frot 77460 Souppes-sur-Loing': [2.732268, 48.184227], 'Rue Léon Avoine 50620 Saint-Fromond': [-1.103616, 49.219704], 'Avenue Léon Blum 31270 Frouzins': [1.32582, 43.521926]}


In [11]:
# find the circonscription for each result
for label, coords in results_dict.items():
    point = Point(coords[0], coords[1])
    for i in range(len(gpf)):
        polygon = gpf.loc[i, 'geometry']
        if polygon.contains(point):
            print(f"The point '{label}' is inside the polygon with index {i}. The circonscription is {gpf.loc[i, 'libelle']}.")
            break

The point '30 Rue Léon Frot 75011 Paris' is inside the polygon with index 71. The circonscription is Paris - 6e circonscription.
The point 'Rue Léon Frot 77460 Souppes-sur-Loing' is inside the polygon with index 453. The circonscription is Seine-et-Marne - 2e circonscription.
The point 'Rue Léon Avoine 50620 Saint-Fromond' is inside the polygon with index 425. The circonscription is Manche - 1re circonscription.
The point 'Avenue Léon Blum 31270 Frouzins' is inside the polygon with index 25. The circonscription is Haute-Garonne - 7e circonscription.
